In [1]:
from pathlib import Path

import pandas as pd
import networkx as nx

import numpy as np
import pandas as pd
import networkx as nx

In [2]:
# ------------------------------------------------------------------
# Parameters
# ------------------------------------------------------------------

# To avoid path explosion, start with a modest cutoff.
# Set to None to allow arbitrary path lengths.
PATH_CUTOFF = 8

In [3]:
# ------------------------------------------------------------------
# Load data
# ------------------------------------------------------------------

csv_file = Path("confidence_rank_unique.csv")  # Change if needed

df = pd.read_csv(csv_file)

In [4]:
# Keep only retained edges
df = df[df["is_kept"] == "KEPT"].copy()

print(f"Kept edges: {len(df)}")

Kept edges: 291


In [5]:
# ------------------------------------------------------------------
# Build graph
# ------------------------------------------------------------------

G = nx.DiGraph()

attribute_columns = [
    c for c in df.columns
    if c not in {"Regulator", "Target", "is_kept"}
]

for _, row in df.iterrows():
    attrs = {c: row[c] for c in attribute_columns}
    G.add_edge(row["Regulator"], row["Target"], **attrs)

print(f"Nodes: {G.number_of_nodes()}")
print(f"Edges: {G.number_of_edges()}")

Nodes: 106
Edges: 291


In [6]:
# ------------------------------------------------------------------
# Sign conversion helpers
# ------------------------------------------------------------------

SIGN_TO_NUM = {
    "positive": 1,
    "negative": -1,
}

NUM_TO_SIGN = {
    1: "positive",
    -1: "negative",
}

In [7]:
# ------------------------------------------------------------------
# Helper
# ------------------------------------------------------------------

def replacement_paths(G, source, target, cutoff=None):
    """Generate replacement paths for a removed edge."""

    if source != target:
        yield from nx.all_simple_paths(
            G,
            source=source,
            target=target,
            cutoff=cutoff,
        )
    else:
        for succ in G.successors(source):
            for path in nx.all_simple_paths(
                G,
                source=succ,
                target=source,
                cutoff=cutoff-1,
            ):
                yield [source] + path

In [8]:
# ------------------------------------------------------------------
# Analyze removed edges
# ------------------------------------------------------------------

removed_df = pd.read_csv(csv_file)
removed_df = removed_df[removed_df["is_kept"] == "TAKEN_OUT"].copy()

results = []
all_path_stats = {}

for idx, row in removed_df.iterrows():

    source = row["Regulator"]
    target = row["Target"]
    original_sign = row["Sign"]
    original_rank = row["Confidence Rank"]

    print(f"Analyzing removed edge: {source} -> {target}")

    # --------------------------------------------------------------
    # Enumerate all replacement paths
    # --------------------------------------------------------------

    path_stats = []

    for path in replacement_paths(
        G,
        source=source,
        target=target,
        cutoff=PATH_CUTOFF,
    ):

        edges = list(zip(path[:-1], path[1:]))

        attrs = [G[u][v] for u, v in edges]

        length = len(edges)

        rank = max(a["Confidence Rank"] for a in attrs)

        sign_num = 1
        for a in attrs:
            sign_num *= SIGN_TO_NUM[a["Sign"]]

        path_stats.append({
            "path": path,
            "length": length,
            "rank": rank,
            "sign_num": sign_num,
            "sign": NUM_TO_SIGN[sign_num],
        })

    all_path_stats[(source, target)] = path_stats

    # --------------------------------------------------------------
    # Require at least one alternative path
    # --------------------------------------------------------------

    if len(path_stats) == 0:
        raise RuntimeError(
            f"No alternative path found for removed edge {source} -> {target}. "
            "This should not happen."
        )

    # --------------------------------------------------------------
    # Select best-rank paths
    # --------------------------------------------------------------

    best_rank = min(p["rank"] for p in path_stats)

    best_paths = [p for p in path_stats if p["rank"] == best_rank]

    lengths = [p["length"] for p in best_paths]
    sign_nums = [p["sign_num"] for p in best_paths]
    sign_set = {p["sign"] for p in best_paths}

    has_positive = "positive" in sign_set
    has_negative = "negative" in sign_set

    avg_sign = float(np.mean(sign_nums))

    missing_original_sign = original_sign not in sign_set

    results.append({
        "Regulator": source,
        "Target": target,
        "Original Sign": original_sign,
        "Original Rank": original_rank,
        "Best Rank": best_rank,
        "Number Best Paths": len(best_paths),
        "Min Length": int(np.min(lengths)),
        "Max Length": int(np.max(lengths)),
        "Avg Length": float(np.mean(lengths)),
        "Has Positive": has_positive,
        "Has Negative": has_negative,
        "Avg Sign": avg_sign,
        "Missing Original Sign": missing_original_sign,
        "Worse Than Original": best_rank > original_rank, 
    })

Analyzing removed edge: BCL6 -> RORGT
Analyzing removed edge: CD4 -> CD4
Analyzing removed edge: CD4 -> RUNX3
Analyzing removed edge: CD8 -> CD8
Analyzing removed edge: CD8 -> THPOK
Analyzing removed edge: CMAF -> IL13
Analyzing removed edge: CMAF -> IL5
Analyzing removed edge: FOXP3 -> FOXP3
Analyzing removed edge: FOXP3 -> IFNG
Analyzing removed edge: FOXP3 -> IFNG_2
Analyzing removed edge: FOXP3 -> IL13
Analyzing removed edge: FOXP3 -> IL17
Analyzing removed edge: FOXP3 -> IL4
Analyzing removed edge: FOXP3 -> IL5
Analyzing removed edge: FOXP3 -> TBET
Analyzing removed edge: FOXP3 -> TBET_2
Analyzing removed edge: FOXP3 -> TGFB
Analyzing removed edge: FOXP3 -> THPOK
Analyzing removed edge: GATA3 -> FOXP3
Analyzing removed edge: GATA3 -> IL10
Analyzing removed edge: GATA3 -> IL12R
Analyzing removed edge: IFNG -> GATA3
Analyzing removed edge: IFNG -> IFNG
Analyzing removed edge: IFNG -> IL10
Analyzing removed edge: IFNG -> IL2
Analyzing removed edge: IFNG -> IL21
Analyzing removed edge

In [9]:
# ------------------------------------------------------------------
# Result table
# ------------------------------------------------------------------

analysis_df = pd.DataFrame(results)

print()
print(f"Analyzed {len(analysis_df)} removed edges.")
print(f"Flagged edges: {analysis_df['Missing Original Sign'].sum()+analysis_df['Worse Than Original'].sum()}")

analysis_df.head()


Analyzed 139 removed edges.
Flagged edges: 2


,Regulator,Target,Original Sign,Original Rank,Best Rank,Number Best Paths,Min Length,Max Length,Avg Length,Has Positive,Has Negative,Avg Sign,Missing Original Sign,Worse Than Original
0,BCL6,RORGT,negative,2,1,2311,2,8,7.519256,True,True,-0.058416,False,False
1,CD4,CD4,positive,4,2,3,2,4,3.000000,True,False,1.000000,False,False
2,CD4,RUNX3,negative,3,2,2,2,3,2.500000,False,True,-1.000000,False,False
3,CD8,CD8,positive,4,2,41,2,8,7.512195,True,True,0.853659,False,False
4,CD8,THPOK,negative,3,2,214,2,8,7.747664,True,True,-0.457944,False,False


In [10]:
missing_original = analysis_df[
    analysis_df["Missing Original Sign"]
]

print(missing_original)

    Regulator Target Original Sign  Original Rank  Best Rank  \
16      FOXP3   TGFB      positive              3          1   
138    TGFB_e   TGFB      positive              2          1   

     Number Best Paths  Min Length  Max Length  Avg Length  Has Positive  \
16                   1           2           2         2.0         False   
138                  4           5           8         6.5         False   

     Has Negative  Avg Sign  Missing Original Sign  Worse Than Original  
16           True      -1.0                   True                False  
138          True      -1.0                   True                False  


In [11]:
worse_than_original = analysis_df[
    analysis_df["Worse Than Original"]]

print(worse_than_original)

Empty DataFrame
Columns: [Regulator, Target, Original Sign, Original Rank, Best Rank, Number Best Paths, Min Length, Max Length, Avg Length, Has Positive, Has Negative, Avg Sign, Missing Original Sign, Worse Than Original]
Index: []


In [12]:
interesting = analysis_df[analysis_df["Missing Original Sign"] | (analysis_df["Worse Than Original"])].copy()

expanded_results = []

for _, row in interesting.iterrows():

    source = row["Regulator"]
    target = row["Target"]
    original_sign = row["Original Sign"]
    original_rank = row["Original Rank"]

    path_stats = all_path_stats[(source, target)]
    matching = [p for p in path_stats if p["sign"] == original_sign]

    if len(matching) == 0:
        expanded_results.append({
            "Regulator": source,
            "Target": target,
            "Original Sign": original_sign,
            "Matching Sign Exists": False,
        })
        continue

    best_rank = min(p["rank"] for p in matching)
    best = [p for p in matching if p["rank"] == best_rank]

    lengths = [p["length"] for p in best]

    expanded_results.append({
        "Regulator": source,
        "Target": target,
        "Original Sign": original_sign,
        "Original Rank": original_rank,
        "Matching Sign Exists": True,
        "Best Matching Rank": best_rank,
        "Number Best Matching Paths": len(best),
        "Min Matching Length": min(lengths),
        "Max Matching Length": max(lengths),
        "Avg Matching Length": np.mean(lengths),
        "Worse Than Original": best_rank > original_rank,
    })

expanded_df = pd.DataFrame(expanded_results)

In [13]:
expanded_df

,Regulator,Target,Original Sign,Original Rank,Matching Sign Exists,Best Matching Rank,Number Best Matching Paths,Min Matching Length,Max Matching Length,Avg Matching Length,Worse Than Original
0,FOXP3,TGFB,positive,3,True,4,38,5,8,7.789474,True
1,TGFB_e,TGFB,positive,2,True,2,5,7,8,7.600000,False
